In [3]:
import pandas as pd
import numpy as np

# Setting a seed for reproducibility
np.random.seed(42)

# Sample size
n_customers = 1143

# Generate random data with logical influences
customers = pd.DataFrame({
    'CustomerID': range(1, n_customers + 1),
    'Age': np.random.randint(18, 70, size=n_customers),
    'Income': np.random.normal(loc=50000, scale=15000, size=n_customers),  # Income might correlate with spending
    'Tenure': np.random.poisson(5, size=n_customers),  # Poisson distribution for tenure
    'Education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], p=[0.3, 0.4, 0.2, 0.1], size=n_customers),
    'Industry': np.random.choice(['Technology', 'Healthcare', 'Finance', 'Education', 'Entertainment'], size=n_customers),
    'Geographic Location': np.random.choice(['North America', 'Europe', 'Asia', 'South America', 'Australia'], size=n_customers),
})

# Adding the effect of attributes on churn likelihood
customers['Churn_Risk'] = np.random.binomial(1, p=customers['Tenure'].apply(lambda x: 0.1 if x > 5 else 0.3))

# Creating a cohort based on the year of first purchase
# Assuming the dataset starts from 2010 to 2019
customers['Cohort'] = pd.to_datetime(np.random.choice(pd.date_range(start='2018-01-01', periods=72, freq='M'), size=n_customers))

customers.to_csv('data/tlacuachitos_vip_customers_data.csv', index=False)
print(customers.head())

   CustomerID  Age        Income  Tenure    Education       Industry  \
0           1   56  52752.677346       3       Master     Technology   
1           2   69  55297.364348       6     Bachelor     Technology   
2           3   46  57978.753383       3     Bachelor        Finance   
3           4   32  60445.266900       3  High School      Education   
4           5   60  57741.870929       5     Bachelor  Entertainment   

  Geographic Location  Churn_Risk     Cohort  
0              Europe           1 2023-08-31  
1       South America           0 2021-08-31  
2              Europe           1 2019-05-31  
3       South America           1 2021-02-28  
4                Asia           0 2018-10-31  


In [5]:
import numpy as np
import pandas as pd

# Assuming the customer DataFrame 'customers' is already created and available

# Seed for reproducibility
np.random.seed(42)

# Generate transactions data
transactions = []

for index, row in customers.iterrows():
    # Enhancing purchase frequency with geographic and age factors
    geo_frequency_factor = {'North America': 1.1, 'Europe': 1.0, 'Asia': 0.9, 'South America': 0.8, 'Australia': 1.2}
    age_frequency_factor = 0.05 * (row['Age'] // 10 - 2)  # Incrementally adjust frequency based on age decade

    n_purchases = np.random.poisson(lam=2 + row['Tenure'] / 5 + geo_frequency_factor[row['Geographic Location']] + age_frequency_factor)  # Base frequency modified by tenure and demographic factors
    
    for _ in range(n_purchases):
        amount = np.random.gamma(shape=2, scale=100 + row['Income'] / 2000)  # Amount influenced by income
        
        # Adding variability based on education level
        education_modifier = {'High School': 0.9, 'Bachelor': 1.0, 'Master': 1.1, 'PhD': 1.2}
        amount *= education_modifier[row['Education']]
        
        # Add industry cyclicality
        industry_cycle = {'Technology': 1.2, 'Healthcare': 1.0, 'Finance': 1.1, 'Education': 0.9, 'Entertainment': 1.3}
        amount *= industry_cycle[row['Industry']]
        
        # Random effect for seasonal or unexpected variations
        seasonal_effect = np.cos(np.pi * np.random.rand())
        amount *= (1 + seasonal_effect * 0.1)
        
        transaction_date = row['Cohort'] + pd.DateOffset(months=int(np.random.exponential(scale=12)))
        
        transactions.append({
            'CustomerID': row['CustomerID'],
            'TransactionDate': transaction_date,
            'TransactionAmount': amount
        })

# Convert the list of transactions to a DataFrame
transactions_df = pd.DataFrame(transactions)
transactions_df['TransactionDate'] = pd.to_datetime(transactions_df['TransactionDate']).dt.date
transactions_df = transactions_df[transactions_df['TransactionDate'] < pd.to_datetime('2024-09-01')]

transactions_df.to_csv('data/tlacuachitos_vip_transactions.csv', index=False)

print(transactions_df.head())
print(transactions_df.info())

   CustomerID TransactionDate  TransactionAmount
1           1      2023-10-31         518.444092
2           1      2024-07-31         353.796197
3           1      2024-01-31          38.206591
4           1      2024-06-30         724.929423
6           2      2022-02-28         145.616000
<class 'pandas.core.frame.DataFrame'>
Int64Index: 4346 entries, 1 to 4721
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         4346 non-null   int64  
 1   TransactionDate    4346 non-null   object 
 2   TransactionAmount  4346 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 135.8+ KB
None


/var/folders/cb/xsxpxpyd055_xtd08vrv79_40000gn/T/ipykernel_8134/2294502786.py:45: FutureWarning: Comparison of Timestamp with datetime.date is deprecated in order to match the standard library behavior. In a future version these will be considered non-comparable. Use 'ts == pd.Timestamp(date)' or 'ts.date() == date' instead.
  transactions_df = transactions_df[transactions_df['TransactionDate'] < pd.to_datetime('2024-09-01')]


In [13]:
transactions_df['TransactionDate'].max()

datetime.date(2024, 8, 31)